In [ ]:
import tensorflow as tf
import numpy as np
import tensorflow_hub as hub
import pandas as pd
from transformers import BertTokenizer

import builtins
import os

seq_len= 512

c:\Users\vanes\miniconda3\envs\nlp_env\Lib\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [8]:

df1= pd.read_excel('datos_sin procesar/concat/23-10-2022.xlsx')
initial_df= pd.concat([df1])
initial_df=initial_df.loc[initial_df['tipoPost']!='no']

#rename column text to post_description and tipoPost to label
initial_df.rename(columns={'text': 'post_description', 'tipoPost': 'label'}, inplace=True)
#replace label "encontró" with "found" and "Busca" with "searching"
initial_df['label'] = initial_df['label'].replace({'encontró': 'found', 'Busca': 'searching'})

initial_df

,post_description,label
0,Perro perdido Puerto Real Cabo Rojo! Ayúdenme ...,found
1,"Encontrado perro en la 9 con 50, es un perro m...",found
2,Ayúdenme a compartir para ver si alguien recon...,found
3,"Difundir por favor, perro macho encontrado en ...",found
4,"Perro perdido, este peludito fue recogido sin ...",found
...,...,...
552,Hola buenas noches mi perro se perdió no ha re...,searching
553,Hola alguien que viva aquí por paseos de sol ?...,searching
554,"AMIGOS, FAVOR DIFUNDIR A FIN DE ENCONTRAR A PR...",found
555,Buscamos a este perrito que se perdió en subac...,searching


In [9]:
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased') #
tokens = tokenizer(
     initial_df['post_description'].tolist(), 
     max_length=seq_len,
     truncation=True, 
     padding='max_length', 
     add_special_tokens=True, 
     return_tensors='np'
)
#tokens.keys()
tokens.input_ids[0]

array([  101, 11982, 10567, 59321, 15968, 12384, 39021, 73631,   106,
       77603, 65579, 10136, 10627,   169, 28974, 10164, 10850, 14652,
         102,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,

In [ ]:
# Crear directorio si no existe
os.makedirs('datos_tokenizados/23-10-2022', exist_ok=True)

with builtins.open ('datos_tokenizados/23-10-2022/posts-xids.npy', 'wb') as f:
     np.save(f, tokens['input_ids'])
with builtins.open ('datos_tokenizados/23-10-2022/posts-xmask.npy', 'wb') as f:
     np.save(f, tokens['attention_mask'])

In [15]:
arr = initial_df['label'].values
#ONE HOT ENCODING

initial_df.loc[initial_df['label']=='found']=1
initial_df.loc[initial_df['label']=='searching']=0

initial_df['label'].value_counts()
#1: user found a dog
#0: user is searching for a dog

label
1    298
0    259
Name: count, dtype: int64

In [16]:
arr = initial_df['label'].values
labels = np.zeros((arr.size, arr.max()+1))
arr=arr.astype('int64')
labels[np.arange(arr.size), arr] = 1
labels

array([[0., 1.],
       [0., 1.],
       [0., 1.],
       ...,
       [0., 1.],
       [1., 0.],
       [1., 0.]], shape=(557, 2))

In [17]:
with open('datos_tokenizados/23-10-2022/posts-labels.npy','wb')as f:
     np.save(f,labels)